# Tamil Nadu Floor Plan — Model Training
## Upload floor_plan_samples.parquet when prompted in Cell 2

In [ ]:
!pip install xgboost shap pyarrow --quiet

from google.colab import files
print("Upload floor_plan_samples.parquet from D:\\layout_project\\training_data\\")
uploaded = files.upload()
print("Upload complete.")

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_parquet("floor_plan_samples.parquet")

print(f"Shape: {df.shape}")
print(f"Valid plans: {df['is_valid'].sum()} ({df['is_valid'].mean()*100:.1f}%)")
print(f"Invalid plans: {(df['is_valid']==0).sum()}")
print()

SCALE_POS_WEIGHT = round((df['is_valid']==0).sum() / df['is_valid'].sum(), 2)
print(f"scale_pos_weight: {SCALE_POS_WEIGHT}")
print()

viol_cols  = [c for c in df.columns if c.startswith('viol_')]
score_cols = [c for c in df.columns if c.startswith('score_')]

print("Violation rates:")
for col in viol_cols:
    print(f"  {col}: {df[col].mean()*100:.1f}%")

print("\nScore means:")
for col in score_cols:
    print(f"  {col}: {df[col].mean():.3f}")

In [ ]:
# Feature columns = everything except labels and string columns
EXCLUDE = set(viol_cols + score_cols + ['is_valid', 'error_type'])
FEATURE_COLS = [c for c in df.columns
                if c not in EXCLUDE
                and df[c].dtype != object]

print(f"Feature columns: {len(FEATURE_COLS)}")

X = df[FEATURE_COLS].astype(float).fillna(-1.0)
y_valid  = df['is_valid'].astype(int)
y_scores = df[score_cols].astype(float)
y_viols  = df[viol_cols].astype(int)

from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y_valid,
    test_size=0.20,
    random_state=42,
    stratify=y_valid
)
print(f"Train: {len(X_tr)}  Test: {len(X_te)}")
print(f"Train valid%: {y_tr.mean()*100:.1f}%")
print(f"Test valid%:  {y_te.mean()*100:.1f}%")

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import (classification_report, roc_auc_score,
                              confusion_matrix)
import joblib

clf = XGBClassifier(
    n_estimators=500,
    max_depth=7,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=SCALE_POS_WEIGHT,
    eval_metric='auc',
    early_stopping_rounds=40,
    random_state=42,
    n_jobs=-1,
    verbosity=1,
)

clf.fit(
    X_tr, y_tr,
    eval_set=[(X_te, y_te)],
    verbose=50,
)

y_pred = clf.predict(X_te)
y_prob = clf.predict_proba(X_te)[:, 1]

print("\n=== CONSTRAINT SCORER RESULTS ===")
print(classification_report(y_te, y_pred,
      target_names=['invalid','valid']))
print(f"ROC-AUC: {roc_auc_score(y_te, y_prob):.4f}")

cm = confusion_matrix(y_te, y_pred)
print(f"\nConfusion matrix:")
print(f"  True Invalid:  {cm[0,0]:>6}  False Valid: {cm[0,1]:>6}")
print(f"  False Invalid: {cm[1,0]:>6}  True Valid:  {cm[1,1]:>6}")

joblib.dump(clf, "constraint_scorer.pkl")
print("\nSaved: constraint_scorer.pkl")

In [ ]:
import matplotlib.pyplot as plt

feat_imp = pd.Series(
    clf.feature_importances_,
    index=FEATURE_COLS
).sort_values(ascending=False)

print("Top 20 most important features:")
for feat, imp in feat_imp.head(20).items():
    bar = '█' * int(imp * 200)
    print(f"  {feat:<35} {imp:.4f}  {bar}")

In [ ]:
import tensorflow as tf
from tensorflow import keras

DIM_INPUT_COLS = [
    'plot_w', 'plot_d', 'plot_area',
    'net_w', 'net_d', 'net_area',
    'bhk', 'facing_code', 'climate_code'
]

DIM_TARGET_COLS = [
    c for c in df.columns
    if (c.endswith('_w') or c.endswith('_d') or c.endswith('_area'))
    and not c.startswith(('plot_', 'net_', 'viol_', 'score_'))
]

print(f"Dimension model: {len(DIM_INPUT_COLS)} inputs → {len(DIM_TARGET_COLS)} outputs")

X_dim = df[DIM_INPUT_COLS].astype(float)
y_dim = df[DIM_TARGET_COLS].astype(float)

# Replace -1.0 sentinel (absent rooms) with NaN then fill with 0
y_dim = y_dim.replace(-1.0, np.nan)
valid_mask = y_dim.notna().all(axis=1)
X_dim_clean = X_dim[valid_mask].reset_index(drop=True)
y_dim_clean = y_dim[valid_mask].fillna(0.0).reset_index(drop=True)

print(f"Clean samples for dim model: {len(X_dim_clean)}")

X_d_tr, X_d_te, y_d_tr, y_d_te = train_test_split(
    X_dim_clean, y_dim_clean,
    test_size=0.20, random_state=42
)

model = keras.Sequential([
    keras.layers.Input(shape=(len(DIM_INPUT_COLS),)),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.1),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(len(DIM_TARGET_COLS), activation='relu'),
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='huber',
    metrics=['mae']
)
model.summary()

callbacks = [
    keras.callbacks.EarlyStopping(
        patience=20,
        restore_best_weights=True,
        monitor='val_loss'
    ),
    keras.callbacks.ReduceLROnPlateau(
        patience=8, factor=0.5,
        min_lr=1e-5, verbose=1
    ),
]

history = model.fit(
    X_d_tr, y_d_tr,
    epochs=200,
    batch_size=512,
    validation_data=(X_d_te, y_d_te),
    callbacks=callbacks,
    verbose=1,
)

loss, mae = model.evaluate(X_d_te, y_d_te, verbose=0)
print(f"\nRoom dimension model MAE: {mae:.4f} meters")
print(f"(Target: MAE < 0.40m means room dims accurate to 40cm)")

model.save("room_dimensions.h5")
print("Saved: room_dimensions.h5")

In [ ]:
import shap

print("Computing SHAP values on 500 test samples...")
explainer = shap.TreeExplainer(clf)
shap_vals = explainer.shap_values(X_te.iloc[:500])

if isinstance(shap_vals, list):
    sv = shap_vals[1]
else:
    sv = shap_vals

shap.summary_plot(
    sv, X_te.iloc[:500],
    plot_type='bar',
    max_display=15,
    show=True
)

joblib.dump(explainer, "shap_explainer.pkl")
print("Saved: shap_explainer.pkl")

In [ ]:
print("=== SANITY CHECK ===\n")

# Test 1: constraint_scorer on 5 samples
clf2 = joblib.load("constraint_scorer.pkl")
sample = X_te.iloc[:5]
preds = clf2.predict(sample)
probs = clf2.predict_proba(sample)[:, 1]
actual = y_te.iloc[:5].values
print("Constraint scorer (5 samples):")
for i in range(5):
    print(f"  actual={actual[i]}  pred={preds[i]}  "
          f"prob_valid={probs[i]:.3f}")

# Test 2: dimension model on a 12x15 2BHK
print("\nDimension model (12x15 2BHK North Composite):")
test_input = np.array([[12.0, 15.0, 180.0, 9.5, 12.5, 118.75,
                         2, 0, 2]])
pred_dims = model.predict(test_input, verbose=0)[0]
for col, val in zip(DIM_TARGET_COLS, pred_dims):
    print(f"  {col}: {val:.2f}m")

# Test 3: SHAP on 1 sample
print("\nSHAP top features for 1 test sample:")
exp2 = joblib.load("shap_explainer.pkl")
sv_one = exp2.shap_values(X_te.iloc[:1:])
if isinstance(sv_one, list):
    sv_one = sv_one[1]
top_idx = np.argsort(np.abs(sv_one[0]))[::-1][:5]
for idx in top_idx:
    print(f"  {FEATURE_COLS[idx]}: shap={sv_one[0][idx]:.4f}")

print("\nALL MODELS OK")

In [ ]:
from google.colab import files

print("Downloading 3 model files...")
print("Save all three to D:\\layout_project\\models\\")
files.download("constraint_scorer.pkl")
files.download("room_dimensions.h5")
files.download("shap_explainer.pkl")
print("Done. Place all files in models\\ folder.")